# model matrix objective loss generation v1

目标：验证 `src/model.matrix` 的 QuantBench 矩阵生成端会和 `objective_scores.csv`、`promotion_scores.csv` 同步写出 `objective_losses.json`，并通过 manifest 进入 dashboard export。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.dashboard import write_model_matrix_dashboard_export
from src.model.matrix import run_quantbench_matrix_smoke

CARD = Path('research/model_matrix_objective_loss_generation_v1')
MATRIX_CARD = CARD / 'matrix_card'
DASHBOARD_EXPORT = CARD / 'dashboard_export.json'
MODEL_NAMES = ('ridge_mse', 'random_forest_clf', 'gcn_mse')

CARD.mkdir(parents=True, exist_ok=True)
MATRIX_CARD.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'matrix_card': str(MATRIX_CARD),
    'model_names': MODEL_NAMES,
}

{'card': 'research/model_matrix_objective_loss_generation_v1',
 'matrix_card': 'research/model_matrix_objective_loss_generation_v1/matrix_card',
 'model_names': ('ridge_mse', 'random_forest_clf', 'gcn_mse')}

## 运行小型 QuantBench matrix

这里用 3 个模型做 smoke evidence，验证生成端 contract；完整模型数量由测试和全量矩阵 notebook 继续覆盖。

In [2]:
outputs = run_quantbench_matrix_smoke(
    card_dir=MATRIX_CARD,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=MODEL_NAMES,
)

objective_losses = json.loads((MATRIX_CARD / 'objective_losses.json').read_text(encoding='utf-8'))
analysis_manifest = json.loads((MATRIX_CARD / 'analysis_manifest.json').read_text(encoding='utf-8'))
artifact_manifest = json.loads((MATRIX_CARD / 'artifact_manifest.json').read_text(encoding='utf-8'))

assert outputs.objective_losses['schema_version'] == 'objective.loss.v1'
assert objective_losses == outputs.objective_losses
assert objective_losses['reward_surface'] == 'retired'
assert analysis_manifest['paths']['objective_losses'] == 'objective_losses.json'
assert artifact_manifest['paths']['objective_losses'] == 'objective_losses.json'
{
    'model_count': outputs.summary['model_count'],
    'loss_schema': objective_losses['schema_version'],
    'reward_surface': objective_losses['reward_surface'],
    'loss_keys': [key for key in ('mse_loss', 'classification_loss', 'ic_loss', 'ranking_loss') if key in objective_losses],
    'analysis_manifest_has_loss': analysis_manifest['paths'].get('objective_losses'),
}

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/numpy/lib/_function_ba

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'model_count': 3,
 'loss_schema': 'objective.loss.v1',
 'reward_surface': 'retired',
 'loss_keys': ['mse_loss', 'classification_loss', 'ic_loss', 'ranking_loss'],
 'analysis_manifest_has_loss': 'objective_losses.json'}

## 写出 dashboard export

dashboard 不读取散落 sidecar，只通过 matrix card 的 manifest 读取 loss。

In [3]:
dashboard_payload = write_model_matrix_dashboard_export(
    MATRIX_CARD,
    output_path=DASHBOARD_EXPORT,
    top_n=3,
)
saved_dashboard = json.loads(DASHBOARD_EXPORT.read_text(encoding='utf-8'))
assert saved_dashboard['manifest_source']['kind'] == 'analysis_manifest'
assert saved_dashboard['objective']['losses']['schema_version'] == 'objective.loss.v1'
assert saved_dashboard['objective']['losses']['reward_surface'] == 'retired'
assert saved_dashboard['artifacts']['objective_losses'].endswith('objective_losses.json')

run_summary = {
    'schema_version': 'research.model_matrix_objective_loss_generation.v1',
    'status': 'completed',
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(DASHBOARD_EXPORT),
    'model_names': list(MODEL_NAMES),
    'model_count': outputs.summary['model_count'],
    'top_model': outputs.summary['top_model'],
    'objective_loss_schema_version': objective_losses['schema_version'],
    'metric_source_schema_version': objective_losses['metric_source_schema_version'],
    'reward_surface': objective_losses['reward_surface'],
    'optimization_direction': objective_losses['optimization_direction'],
    'loss_component_keys': [key for key in ('mse_loss', 'classification_loss', 'ic_loss', 'ranking_loss') if key in objective_losses],
    'rank_group_count': objective_losses['rank_group_count'],
    'analysis_manifest_has_loss': analysis_manifest['paths'].get('objective_losses') == 'objective_losses.json',
    'artifact_manifest_has_loss': artifact_manifest['paths'].get('objective_losses') == 'objective_losses.json',
    'dashboard_export_has_loss': saved_dashboard['objective']['losses']['schema_version'] == 'objective.loss.v1',
    'conclusion': 'QuantBench matrix 生成端已同步写出 objective.loss.v1，并通过 manifest 进入 dashboard export。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_summary

{'schema_version': 'research.model_matrix_objective_loss_generation.v1',
 'status': 'completed',
 'matrix_card': 'research/model_matrix_objective_loss_generation_v1/matrix_card',
 'dashboard_export': 'research/model_matrix_objective_loss_generation_v1/dashboard_export.json',
 'model_names': ['ridge_mse', 'random_forest_clf', 'gcn_mse'],
 'model_count': 3,
 'top_model': 'random_forest_clf',
 'objective_loss_schema_version': 'objective.loss.v1',
 'metric_source_schema_version': 'objective.evaluation.v1',
 'reward_surface': 'retired',
 'optimization_direction': 'minimize',
 'loss_component_keys': ['mse_loss',
  'classification_loss',
  'ic_loss',
  'ranking_loss'],
 'rank_group_count': 0,
 'analysis_manifest_has_loss': True,
 'artifact_manifest_has_loss': True,
 'dashboard_export_has_loss': True,
 'conclusion': 'QuantBench matrix 生成端已同步写出 objective.loss.v1，并通过 manifest 进入 dashboard export。'}

## 中文结论

- 做了什么：运行 3 模型 QuantBench smoke matrix，验证 `src/model.matrix` 生成 `objective_losses.json`，并在 `analysis_manifest.json`、`artifact_manifest.json` 中暴露该 sidecar。
- 核心指标：`objective.loss.v1` 保留 `mse_loss`、`classification_loss`、`ic_loss`、`ranking_loss`、`rank_group_count`，`reward_surface=retired`，`optimization_direction=minimize`。
- regime 差异：本实验不比较策略优劣，只验证生成端 contract；regime/window 结果仍由 matrix card 的 `regime_metrics.csv`、`window_metrics.csv` 和 dashboard export 承接。
- 是否晋升：这是框架能力证据，晋升到 `src/model.matrix` 的稳定输出 contract；不是生产策略晋升。
- 风险：本 notebook 是 3 模型 smoke evidence；完整 QuantBench 模型面仍由单元测试和后续全量矩阵 notebook 维护。
- 下一步：把 dashboard 页面层增加 objective loss 对比区域，或在 schedule notification 中加入 loss schema/状态摘要。